# CO3133 A2 - Food-101 training on Kaggle, **Internet OFF** (M3: every reported run)

Inputs (both private datasets, built on a local machine with `scripts/make_kaggle_bundle.py`):
- `co3113-a2-code`: the repository at one commit (`co3113-a2-code.zip`, incl. `GIT_COMMIT`) + the torchvision
  ImageNet weights of ResNet-50 / EfficientNet-B0 / ConvNeXt-T (`*.pth`). No `git clone`, `pip` or weight download here.
- `food-101-256`: the pre-resized images (`food-101-256.zip`).

1. *Settings → Accelerator:* a GPU (RTX 6000, T4, P100 all work), *Internet: Off*. *Add Input →* both datasets.
2. Optional, to **resume**: also add the output of an earlier session of this notebook; its `outputs/runs` is copied
   back, finished runs are skipped and unfinished ones continue from `last.pt`.
3. Choose `SESSION` below (`"all"`, or `"A"` mandatory / `"B"` optional to split over two sessions or accounts), then
   *Save Version → Save & Run All (Commit)*. Each run only gets the time left before `SESSION_HOURS`, so the session
   ends cleanly before Kaggle's 12 h limit.
4. Download `results_bundle_*.zip` and `checkpoints_food101.zip` from the output tab; unzip both **at the repo root**.


In [ ]:
import glob, os, shutil, subprocess, sys, time, zipfile
T0 = time.time()
REPO = "/kaggle/working/CO3113-A2"
# code: Kaggle may or may not have extracted the zip -> accept both
hits = [h for h in glob.glob("/kaggle/input/**/CO3113-A2/GIT_COMMIT", recursive=True) if "co3113-a2-code" in h]
if not os.path.exists(REPO):
    if hits:
        shutil.copytree(os.path.dirname(hits[0]), REPO)
    else:
        zips = glob.glob("/kaggle/input/**/co3113-a2-code.zip", recursive=True)
        assert zips, "add the co3113-a2-code dataset as input"
        zipfile.ZipFile(zips[0]).extractall("/kaggle/working")
os.chdir(REPO)
print("code at commit", open("GIT_COMMIT").read().strip())

# ImageNet weights -> torch hub cache (torchvision loads them from there instead of downloading)
import torch
hub = os.path.join(torch.hub.get_dir(), "checkpoints")
os.makedirs(hub, exist_ok=True)
for name in ["resnet50-11ad3fa6.pth", "efficientnet_b0_rwightman-7f5810bc.pth", "convnext_tiny-983f1562.pth"]:
    src = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert src, f"{name} missing from the co3113-a2-code dataset"
    shutil.copy(src[0], os.path.join(hub, name))
print(os.listdir(hub))

In [ ]:
# no pip offline: the Kaggle image already ships every dependency of requirements.txt
import numpy, scipy, sklearn, pandas, matplotlib, PIL, yaml, tqdm, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| numpy", numpy.__version__,
      "| sklearn", sklearn.__version__, "| pandas", pandas.__version__)
print(torch.cuda.get_device_name(0), "| CPUs:", os.cpu_count())
!nvidia-smi
!free -g | head -2

In [ ]:
def find_food101(pattern_root):
    """Return every folder under pattern_root that has meta/classes.txt and images/."""
    hits = [os.path.dirname(os.path.dirname(p)) for p in glob.glob(f"{pattern_root}/**/meta/classes.txt", recursive=True)]
    return [h for h in hits if os.path.isdir(os.path.join(h, "images")) and "CO3113-A2" not in h]
roots = [r for r in find_food101("/kaggle/input") if "256" in r]
if not roots:                         # dataset kept as a zip -> extract once (stored zip, ~1 min)
    z = glob.glob("/kaggle/input/**/food-101-256.zip", recursive=True)
    assert z, "add the food-101-256 dataset as input"
    zipfile.ZipFile(z[0]).extractall("/tmp/food101")
    roots = find_food101("/tmp/food101")
IMAGE_ROOT = roots[0]
print("images:", IMAGE_ROOT)

# resume: copy outputs/runs of an earlier session (if added as input)
for prev in glob.glob("/kaggle/input/**/outputs/runs/food101", recursive=True):
    print("restoring", prev)
    shutil.copytree(prev, "outputs/runs/food101", dirs_exist_ok=True)

In [ ]:
# what to run. Order = priority (mandatory first). E3 and P3 use seed 42 only (proposal section 5).
SESSION = "all"            # "all" | "A" (mandatory) | "B" (optional)
SESSION_HOURS = 11.0       # stop starting / continuing runs after this (Kaggle kills the session at 12 h)
SEEDS3 = [42, 123, 2026]
JOBS_A = [(e, s) for e in ["b0_simple_cnn", "p1_resnet50", "p2_efficientnet_b0", "e1_resnet50_linear_probe"] for s in SEEDS3]
JOBS_B = ([("e2_resnet50_label_smoothing", s) for s in SEEDS3]
          + [(e, 42) for e in ["e3_p1_resnet50_frac10", "e3_p1_resnet50_frac25", "e3_p1_resnet50_frac50",
                               "e3_b0_simple_cnn_frac10", "e3_b0_simple_cnn_frac25", "e3_b0_simple_cnn_frac50",
                               "p3_convnext_tiny"]])
JOBS = {"all": JOBS_A + JOBS_B, "A": JOBS_A, "B": JOBS_B}[SESSION]
NUM_WORKERS = os.cpu_count()      # data loading (JPEG decode + augmentation) is the bottleneck on fast GPUs
COMMON = [f"data.image_root={IMAGE_ROOT}", f"data.num_workers={NUM_WORKERS}", "train.show_progress=false"]
print(len(JOBS), "jobs, workers:", NUM_WORKERS)

In [ ]:
# one run_experiments call per job: finished runs are skipped, unfinished ones resumed; each run only gets the
# time left in the session (train.max_hours stops it cleanly after an epoch, keeping last.pt for the next session)
status = []
for exp, seed in JOBS:
    left = SESSION_HOURS - (time.time() - T0) / 3600
    if left < 0.4:
        status.append((exp, seed, "not started (session budget used)"))
        continue
    rc = subprocess.run([sys.executable, "scripts/run_experiments.py", "--models", exp, "--seeds", str(seed),
                         "--set", *COMMON, f"train.max_hours={left - 0.25:.2f}"]).returncode
    done = os.path.exists(f"outputs/runs/food101/{exp}/seed{seed}/eval/test_metrics.json")
    status.append((exp, seed, "finished" if done else f"UNFINISHED (rc {rc}) - rerun to resume"))
for s in status:
    print(*s)

In [ ]:
# free space: last.pt (weights + optimizer state) is only needed by unfinished runs
for f in glob.glob("outputs/runs/food101/*/seed*/checkpoints/last.pt"):
    if os.path.exists(os.path.join(os.path.dirname(os.path.dirname(f)), "eval", "test_metrics.json")):
        os.remove(f)
!python scripts/compare.py
!python scripts/pack_results.py --out-dir /kaggle/working
!python scripts/pack_results.py --checkpoints-only --out-dir /kaggle/working
!ls -lh /kaggle/working